# Capítulo 8 — Conexões e Operações de multi-tabelas: Não podemos todos nos entender?

**Livro:** começa na página **313** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")
from banco import preparar
motor = preparar(8, "gabarito", recomecar=True)  # preparar(..., recomecar=True) volta o capítulo ao início
%reload_ext sql
%config SqlMagic.displaylimit = 60
%config SqlMagic.feedback = 0
%sql motor

## Os interesses especiais (coluna) · PDF p. 317

#### Exercício: Colunas para os interesses

Em `contatos_antigos`, os interesses estão juntos e separados por vírgula. Acrescente as colunas `interesse1`, `interesse2` e `interesse3`.

In [ ]:
%%sql
ALTER TABLE contatos_antigos ADD COLUMN interesse1 VARCHAR(30), ADD COLUMN interesse2 VARCHAR(30),
                             ADD COLUMN interesse3 VARCHAR(30);
SELECT * FROM contatos_antigos;

## UPDATE todos seus interesses · PDF p. 319

#### Exercício: Separando com split_part

Preencha as três colunas com `split_part(interesses, ',', 1)`, `2` e `3` (no MySQL seria `SUBSTRING_INDEX`). Use `NULLIF(..., '')` para que a falta de valor vire `NULL`.

In [ ]:
%%sql
UPDATE contatos_antigos SET
    interesse1 = NULLIF(split_part(interesses, ',', 1), ''),
    interesse2 = NULLIF(split_part(interesses, ',', 2), ''),
    interesse3 = NULLIF(split_part(interesses, ',', 3), '');
SELECT contato_id, primeiro_nome, interesse1, interesse2, interesse3 FROM contatos_antigos ORDER BY contato_id;

## CREATE, SELECT, e INSERT a (quase) o mesmo tempo · PDF p. 321

#### Exercício: CREATE e depois INSERT ... SELECT

Crie `lista_interesses (id SERIAL PRIMARY KEY, interesse VARCHAR(30))` e preencha com os valores distintos de `interesse1` (sem nulos), em ordem.

In [ ]:
%%sql
CREATE TABLE lista_interesses (id SERIAL PRIMARY KEY, interesse VARCHAR(30));
INSERT INTO lista_interesses (interesse)
SELECT DISTINCT interesse1 FROM contatos_antigos WHERE interesse1 IS NOT NULL ORDER BY interesse1;
SELECT * FROM lista_interesses;

## CREATE, SELECT e INSERT ao mesmo tempo · PDF p. 322

#### Exercício: CREATE TABLE ... AS SELECT

Crie numa só instrução a tabela `todos_interesses` com **todos** os interesses distintos das três colunas (junte com `UNION`).

In [ ]:
%%sql
CREATE TABLE todos_interesses AS
SELECT interesse1 AS interesse FROM contatos_antigos WHERE interesse1 IS NOT NULL
UNION SELECT interesse2 FROM contatos_antigos WHERE interesse2 IS NOT NULL
UNION SELECT interesse3 FROM contatos_antigos WHERE interesse3 IS NOT NULL;
SELECT * FROM todos_interesses ORDER BY interesse;

## Pseudônimos de coluna · PDF p. 323

#### Exercício: AS em colunas

Mostre primeiro nome, sobrenome e aniversário de `meus_contatos` com os títulos `nome`, `sobrenome` e `nascimento`.

In [ ]:
%%sql
SELECT primeiro_nome AS nome, sobrenome, aniversario AS nascimento FROM meus_contatos;

## Pseudônimos de tabelas, quem precisa deles? · PDF p. 324

#### Exercício: AS em tabelas

Repita usando o apelido `mc` para a tabela (`FROM meus_contatos AS mc`) e `mc.coluna` no SELECT.

In [ ]:
%%sql
SELECT mc.primeiro_nome AS nome, mc.email FROM meus_contatos AS mc WHERE mc.sexo = 'F';

## Conexão cartesiana · PDF p. 325

#### Exercício: CROSS JOIN

Combine cada garoto com cada brinquedo (`CROSS JOIN`) e conte as linhas. Quanto deveria dar?

In [ ]:
%%sql
SELECT COUNT(*) AS combinacoes,
       (SELECT COUNT(*) FROM garotos) * (SELECT COUNT(*) FROM brinquedos) AS esperado
FROM garotos CROSS JOIN brinquedos;

## A conexão interna em ação: a equijoin · PDF p. 329

#### Exercício: INNER JOIN (equijoin)

Mostre cada garoto com o **seu** brinquedo (`INNER JOIN ... ON g.brinquedo_id = b.brinquedo_id`).

In [ ]:
%%sql
SELECT g.garoto, b.brinquedo
FROM garotos AS g
INNER JOIN brinquedos AS b ON g.brinquedo_id = b.brinquedo_id
ORDER BY g.garoto;

## A conexão interna em ação: a não-equijoin · PDF p. 332

#### Exercício: Não-equijoin

Mostre, para cada garoto, os brinquedos que ele **não** tem (`ON g.brinquedo_id <> b.brinquedo_id`).

In [ ]:
%%sql
SELECT g.garoto, b.brinquedo
FROM garotos AS g
INNER JOIN brinquedos AS b ON g.brinquedo_id <> b.brinquedo_id
ORDER BY g.garoto, b.brinquedo;

## A última conexão interna: a conexão natural · PDF p. 334

#### Exercício: NATURAL JOIN

As duas tabelas têm a coluna `brinquedo_id`. Use `NATURAL JOIN` e compare com a equijoin.

In [ ]:
%%sql
SELECT garoto, brinquedo FROM garotos NATURAL JOIN brinquedos ORDER BY garoto;

## Consultas conectadas? · PDF p. 339

#### Exercício: Várias tabelas de uma vez

Mostre nome, profissão, cidade, UF e estado civil de cada contato juntando `meus_contatos`, `profissao`, `codigos_cep` e `estado_civil`. Depois acrescente os interesses de cada um numa só célula com `string_agg` (via `contato_interesse` e `interesses`).

In [ ]:
%%sql
SELECT mc.primeiro_nome, p.profissao, c.cidade, c.uf, e.status,
       string_agg(i.interesse, ', ' ORDER BY i.interesse) AS interesses
FROM meus_contatos mc
JOIN profissao p ON p.prof_id = mc.prof_id
JOIN codigos_cep c ON c.cep = mc.cep
JOIN estado_civil e ON e.status_id = mc.status_id
LEFT JOIN contato_interesse ci ON ci.contato_id = mc.contato_id
LEFT JOIN interesses i ON i.interesse_id = ci.interesse_id
GROUP BY mc.contato_id, mc.primeiro_nome, p.profissao, c.cidade, c.uf, e.status
ORDER BY mc.contato_id;